<a href="https://colab.research.google.com/github/nicolasvgot-alt/Integraci-n-de-datos-y-prospecci-n/blob/main/Integra_Parcial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este caso de estudio analiza cómo una entidad de salud (EPS) debe manejar el cierre de su sede en Sabaneta de forma estratégica. El reto principal es reubicar a los pacientes con diabetes en otras sucursales sin perder el control de la información. Para esto, el proyecto propone un proceso de integración de datos en dos pasos: primero, usamos la Teoría de la Credibilidad para descubrir qué sedes son más parecidas o tienen más 'afinidad' con la que va a cerrar; y segundo, comparamos dos métodos (el de Valor de Pertenencia y el de Aceptación y Rechazo) para organizar a los pacientes en sus nuevos destinos. Finalmente, el éxito del proceso se mide comparando las estadísticas de los datos (como los promedios y la variabilidad) antes y después de la integración, para asegurar que la atención de los pacientes siga siendo consistente."

In [69]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
# Sabaneta
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos = pd.read_excel(XDB)
datos.head(10)

,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,SAB-0001,196,101,0,321,31.010,0.352,80,0,Sabaneta
1,SAB-0002,93,115,17,366,27.789,1.846,31,0,Sabaneta
2,SAB-0003,95,73,37,135,44.594,1.461,63,1,Sabaneta
3,SAB-0004,94,91,19,191,23.219,1.303,22,0,Sabaneta
4,SAB-0005,189,47,22,600,24.747,0.993,43,1,Sabaneta
5,SAB-0006,185,58,34,619,33.292,0.589,30,0,Sabaneta
6,SAB-0007,108,92,6,284,30.470,1.493,46,1,Sabaneta
7,SAB-0008,191,110,54,435,34.060,1.279,80,1,Sabaneta
8,SAB-0009,196,45,17,490,21.073,2.444,69,0,Sabaneta
9,SAB-0010,163,62,33,834,29.303,1.329,40,1,Sabaneta


In [71]:
## Esta base de datos, es sobre la sucursal de Bello
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos1 = pd.read_excel(XDB,sheet_name=1)
datos1.head(10)

,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,BEL-0001,167,44,50,135,19.375,0.788,49,0,Bello
1,BEL-0002,60,54,12,602,43.115,1.640,37,1,Bello
2,BEL-0003,104,89,13,845,21.230,1.958,35,1,Bello
3,BEL-0004,72,61,12,145,49.781,1.643,25,1,Bello
4,BEL-0005,68,75,58,529,45.160,0.471,79,1,Bello
5,BEL-0006,184,70,41,256,45.977,1.871,22,0,Bello
6,BEL-0007,135,89,44,700,24.707,0.658,56,1,Bello
7,BEL-0008,179,113,9,247,17.449,0.327,40,1,Bello
8,BEL-0009,57,61,12,256,47.579,1.661,47,0,Bello
9,BEL-0010,139,98,21,87,29.262,0.750,55,1,Bello


In [74]:
## Esta base de datos es sobre la sede de Medellín
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos2 = pd.read_excel(XDB,sheet_name=2)
datos2.head(10)

,PatientID,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,MED-0001,14,40,69,58,605,35.901,0.986,56,1,Medellín
1,MED-0002,14,165,112,7,143,29.895,1.377,73,1,Medellín
2,MED-0003,1,150,50,52,132,43.766,1.866,57,0,Medellín
3,MED-0004,2,55,60,56,705,35.141,1.038,43,1,Medellín
4,MED-0005,4,58,43,32,790,20.618,0.681,51,1,Medellín
5,MED-0006,11,65,41,31,588,21.866,2.203,67,1,Medellín
6,MED-0007,11,191,94,55,768,28.684,1.006,23,0,Medellín
7,MED-0008,6,188,76,17,328,34.599,2.009,81,0,Medellín
8,MED-0009,0,61,90,35,171,49.877,1.121,55,1,Medellín
9,MED-0010,9,164,80,38,72,17.791,0.758,60,1,Medellín


In [75]:
## Esta base de datos es sobrre la sede de Envigado
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos3 = pd.read_excel(XDB,sheet_name=3)
datos3.head(10)

,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,ENV-0001,164,81,56,46,26.921,1.984,81,1,Envigado
1,ENV-0002,121,71,15,245,29.110,2.484,22,1,Envigado
2,ENV-0003,187,46,9,609,32.877,2.142,27,0,Envigado
3,ENV-0004,82,81,6,564,49.109,1.293,29,1,Envigado
4,ENV-0005,88,64,51,409,31.105,2.213,40,1,Envigado
5,ENV-0006,121,43,32,229,28.763,0.385,27,1,Envigado
6,ENV-0007,152,119,55,542,20.052,0.580,55,0,Envigado
7,ENV-0008,67,105,17,763,33.969,1.399,81,0,Envigado
8,ENV-0009,92,117,8,265,39.155,1.880,61,1,Envigado
9,ENV-0010,87,64,49,278,30.239,0.809,51,0,Envigado


In [76]:
## Esta base de datos es sobre la sede de Itagui
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos4 = pd.read_excel(XDB,sheet_name=4)
datos4.head(10)

,PatientID,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,ITA-0001,89,39,736,25.563,0.557,59,0,Itagui
1,ITA-0002,99,31,238,49.786,1.190,46,0,Itagui
2,ITA-0003,110,18,593,49.004,1.117,76,0,Itagui
3,ITA-0004,70,47,71,47.335,0.159,23,1,Itagui
4,ITA-0005,40,50,369,34.553,1.018,33,0,Itagui
5,ITA-0006,118,12,730,33.849,0.130,54,1,Itagui
6,ITA-0007,117,2,748,28.181,0.848,45,0,Itagui
7,ITA-0008,120,2,688,39.515,1.995,75,1,Itagui
8,ITA-0009,107,49,729,22.048,1.607,62,1,Itagui
9,ITA-0010,103,13,340,29.793,1.679,75,1,Itagui


In [77]:
## Esta es sobre la sede de Caldas
XDB = '/content/3. Parcial - medical_attention_data-Cambio.xlsx'
datos5 = pd.read_excel(XDB,sheet_name=5)
datos5.head(10)

,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,CAL-0001,166,41,40,31,29.526,1.293,63,0,Caldas
1,CAL-0002,162,55,32,134,15.300,2.205,72,0,Caldas
2,CAL-0003,185,79,29,188,33.307,0.113,24,0,Caldas
3,CAL-0004,74,117,25,553,38.052,1.040,71,1,Caldas
4,CAL-0005,183,95,44,196,29.501,1.694,71,0,Caldas
5,CAL-0006,53,48,16,112,31.665,0.465,54,1,Caldas
6,CAL-0007,196,49,13,335,27.748,0.656,61,1,Caldas
7,CAL-0008,177,54,52,520,36.313,0.852,81,0,Caldas
8,CAL-0009,162,120,23,413,18.959,0.535,77,1,Caldas
9,CAL-0010,124,98,33,363,24.597,0.492,31,1,Caldas


In [82]:
##Cómo se va a evaluar la crredibilidad y aceptación, necesitamos saber si hay algún dato faltante
datos.isna().sum()

,0
PatientID,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Diabetes,0
Branch,0


In [83]:
datos_matrix =datos

In [84]:
datos_matrix.insert(0, 'ID', np.arange(1, len(datos_matrix) + 1))

In [85]:
datos_matrix = datos4.drop('PatientID', axis=1)
datos_matrix.head()

,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch
0,89,39,736,25.563,0.557,59,0,Itagui
1,99,31,238,49.786,1.190,46,0,Itagui
2,110,18,593,49.004,1.117,76,0,Itagui
3,70,47,71,47.335,0.159,23,1,Itagui
4,40,50,369,34.553,1.018,33,0,Itagui


In [86]:
datos_matrix = datos_matrix.drop('Branch', axis=1)
datos_matrix.head()

,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
0,89,39,736,25.563,0.557,59,0
1,99,31,238,49.786,1.190,46,0
2,110,18,593,49.004,1.117,76,0
3,70,47,71,47.335,0.159,23,1
4,40,50,369,34.553,1.018,33,0


In [87]:
corr_matrix = datos_matrix.corr()
corr_matrix

,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
BloodPressure,1.000000,-0.036428,0.004846,-0.018601,0.017753,0.044379,-0.046963
SkinThickness,-0.036428,1.000000,-0.020066,0.028027,-0.035110,-0.028441,-0.006072
Insulin,0.004846,-0.020066,1.000000,-0.001245,0.019521,0.005857,-0.036844
BMI,-0.018601,0.028027,-0.001245,1.000000,-0.066471,-0.013813,-0.038909
DiabetesPedigreeFunction,0.017753,-0.035110,0.019521,-0.066471,1.000000,0.025362,0.008895
Age,0.044379,-0.028441,0.005857,-0.013813,0.025362,1.000000,-0.042508
Diabetes,-0.046963,-0.006072,-0.036844,-0.038909,0.008895,-0.042508,1.000000


In [88]:
datos_matrix.head(10)

,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
0,89,39,736,25.563,0.557,59,0
1,99,31,238,49.786,1.190,46,0
2,110,18,593,49.004,1.117,76,0
3,70,47,71,47.335,0.159,23,1
4,40,50,369,34.553,1.018,33,0
5,118,12,730,33.849,0.130,54,1
6,117,2,748,28.181,0.848,45,0
7,120,2,688,39.515,1.995,75,1
8,107,49,729,22.048,1.607,62,1
9,103,13,340,29.793,1.679,75,1


In [90]:
##Con este codigo conocemos los valores estadisticos por Sucursal, en este caso revisamos la de Sabaneta que es la que se va a cerrar
datos.describe()

,ID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
count,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000
mean,311.000000,115.821256,80.228663,30.400966,422.504026,32.992060,1.314498,51.673108,0.510467
std,179.411538,45.654084,23.709517,17.861256,239.814940,9.894817,0.716244,17.429334,0.500293
min,1.000000,40.000000,40.000000,0.000000,0.000000,15.003000,0.073000,21.000000,0.000000
25%,156.000000,75.000000,59.000000,15.000000,212.000000,24.478000,0.633000,38.000000,0.000000
50%,311.000000,114.000000,80.000000,31.000000,424.000000,32.923000,1.334000,52.000000,1.000000
75%,466.000000,154.000000,101.000000,47.000000,614.000000,41.537000,1.968000,66.000000,1.000000
max,621.000000,200.000000,120.000000,60.000000,849.000000,49.944000,2.492000,81.000000,1.000000


In [91]:
##Resultados estadisticos de Bello
datos1.describe()

,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
count,502.000000,502.000000,502.000000,502.000000,502.00000,502.000000,502.000000,502.000000
mean,119.175299,79.565737,31.049801,421.641434,32.99048,1.271197,52.207171,0.490040
std,47.474598,24.105032,17.746257,241.854460,9.88173,0.719036,18.140748,0.500399
min,41.000000,40.000000,0.000000,1.000000,15.00300,0.070000,21.000000,0.000000
25%,76.250000,59.000000,16.000000,212.000000,24.31550,0.642750,37.000000,0.000000
50%,122.000000,79.500000,32.000000,431.500000,33.45750,1.253000,53.000000,0.000000
75%,157.750000,101.000000,47.000000,628.000000,40.79875,1.886500,69.000000,1.000000
max,200.000000,120.000000,60.000000,850.000000,49.85300,2.496000,81.000000,1.000000


In [92]:
##Resultados estadisticos de Medellín
datos2.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
count,895.000000,895.000000,895.000000,895.000000,895.000000,895.000000,895.000000,895.000000,895.000000
mean,8.170950,122.277095,79.902793,30.345251,421.836872,32.879753,1.262841,51.626816,0.471508
std,5.079771,46.600008,23.171246,17.811288,251.093390,9.976137,0.700308,17.737439,0.499467
min,0.000000,40.000000,40.000000,0.000000,0.000000,15.008000,0.072000,21.000000,0.000000
25%,4.000000,82.000000,60.000000,15.000000,208.000000,24.114000,0.680500,36.000000,0.000000
50%,8.000000,125.000000,80.000000,31.000000,402.000000,33.085000,1.228000,52.000000,0.000000
75%,12.000000,163.000000,100.000000,46.000000,641.000000,41.213000,1.875000,67.000000,1.000000
max,17.000000,200.000000,120.000000,60.000000,849.000000,49.971000,2.498000,81.000000,1.000000


**Sacamos las variables para poder ver la afinidad**

1. De la base de Datos de sabaneta, necesitamos, Edad, Presión de la Sangre, BMI.

In [131]:
##Esta serán las variables que sacamos de la sucursal Sabaneta, se van a diferenciar por el numero de hoja del excel
SkinThickness=datos.iloc[:,4]



,SkinThickness
0,0
1,17
2,37
3,19
4,22
...,...
616,3
617,43
618,10
619,27


In [132]:
# Bello

SkinThickness1=datos1.iloc[:,3]

In [122]:
#mede
SkinThickness2= datos2.iloc[:,4]

,SkinThickness
0,58
1,7
2,52
3,56
4,32
...,...
890,28
891,31
892,20
893,12


In [133]:
SkinThickness3 = datos3.iloc[:,3]

In [134]:
##Datos prom para Itaguí
SkinThickness4=datos4.iloc[:,2]


In [135]:
##Datos prom para Caldas

SkinThickness5=datos5.iloc[:,3]


In [140]:
def caracterizacion(SkinThickness):



  #Se procede con la caracterización de cada una de las variables
  np.set_printoptions(suppress=True)
  NI=10    #Indica el número de clusters
  counts,bin_edges=np.histogram(SkinThickness,bins=NI)
  print("El número de datos por intervalo es:")
  print(counts)
  print("Los intervalos inferiores:")
  print(bin_edges[:-1])
  print("Los intervalos superiores:")
  print(bin_edges[1:])
  XC=(bin_edges[:-1]+bin_edges[1:])/2

  #Se configura la tabla de los datos
  Tabla=np.column_stack((bin_edges[:-1],bin_edges[1:],XC,counts))
  df=pd.DataFrame(Tabla,columns=['LI','LS','XC','ND'])
  df.head(10)

  #Se procede con la estimación de la media
  fr=counts/np.sum(counts)
  u=np.sum(XC*fr)
  sigma2=np.sum(fr*(XC-u)**2)
  sigma=np.sqrt(sigma2)
  Cas=np.sum(fr*(XC-u)**3)/sigma**3
  Kur=(np.sum(fr*(XC-u)**4)/sigma**4)-3

  return u,sigma,Cas,Kur,df

In [141]:
def RiskParameters(SkinThickness):

  NDT=len(SkinThickness)
  PE=np.mean(SkinThickness)
  NPE=np.sum(SkinThickness<PE)
  OpVar=np.percentile(SkinThickness1,99.9)
  NPC=np.sum(SkinThickness>OpVar)
  NPNE=NDT-NPE-NPC

  return OpVar,NPE,NPNE,NPC

In [142]:
uo,sigmao, Caso, Kuo,dfo=caracterizacion(SkinThickness)
print("La media de los datos observados es:", uo)
print("El coeficiente de asimetría es: ", Caso)

El número de datos por intervalo es:
[59 65 57 54 66 65 51 54 72 78]
Los intervalos inferiores:
[ 0.  6. 12. 18. 24. 30. 36. 42. 48. 54.]
Los intervalos superiores:
[ 6. 12. 18. 24. 30. 36. 42. 48. 54. 60.]
La media de los datos observados es: 30.94202898550725
El coeficiente de asimetría es:  -0.03588911256234775


Medidas de tendencia
2. Vamos a evaluar cada medida por base de datos, para ver más o menos cómo estan de similares los datos


In [143]:
#Base de datos Sabaneta
us,sigmas, Cass, Kus,dfs=caracterizacion(SkinThickness)
print("La media de los datos observados es en Sabaneta:", us)
print("El coeficiente de asimetría es en Sabaneta: ", Cass)
OpVars,NPEs,NPNEs,NPCs=RiskParameters(SkinThickness)

#Base de datos de Bello
ub,sigmab, Casb, Kub,dfb=caracterizacion(SkinThickness1)
print("La media de los datos observados es en Bello:", ub)
print("El coeficiente de asimetría es en Bello: ", Casb)
OpVarb,NPEb,NPNEb,NPCb=RiskParameters(SkinThickness1)

#Base de datos de Medellín
um,sigmam, Casm, Kum,dfm=caracterizacion(SkinThickness2)
print("La media de los datos observados es en Medellín:", um)
print("El coeficiente de asimetría es en Medellín: ", Casm)
OpVarm,NPEm,NPNEm,NPCm=RiskParameters(SkinThickness2)

#Base de datos Envigado
ue,sigmae, Case, Kue,dfe=caracterizacion(SkinThickness3)
print("La media de los datos observados es en Envigado:", ue)
print("El coeficiente de asimetría es en Envigado: ", Case)
OpVare,NPEe,NPNEe,NPCe=RiskParameters(SkinThickness3)

#Base de datos Itaguí
ui,sigmai, Casi, Kui,dfi=caracterizacion(SkinThickness4)
print("La media de los datos observados es en Itaguí:", ui)
print("El coeficiente de asimetría es en Itaguí: ", Casi)
OpVari,NPEi,NPNEi,NPCi=RiskParameters(SkinThickness4)

#Base de datos en Caldas
uc,sigmac, Casc, Kuc,dfc=caracterizacion(SkinThickness5)
print("La media de los datos observados es en Caldas:", uc)
print("El coeficiente de asimetría es en Caldas: ", Casc)
OpVarc,NPEc,NPNEc,NPCc=RiskParameters(SkinThickness5)

El número de datos por intervalo es:
[59 65 57 54 66 65 51 54 72 78]
Los intervalos inferiores:
[ 0.  6. 12. 18. 24. 30. 36. 42. 48. 54.]
Los intervalos superiores:
[ 6. 12. 18. 24. 30. 36. 42. 48. 54. 60.]
La media de los datos observados es en Sabaneta: 30.94202898550725
El coeficiente de asimetría es en Sabaneta:  -0.03588911256234775
El número de datos por intervalo es:
[49 42 45 53 47 54 39 51 59 63]
Los intervalos inferiores:
[ 0.  6. 12. 18. 24. 30. 36. 42. 48. 54.]
Los intervalos superiores:
[ 6. 12. 18. 24. 30. 36. 42. 48. 54. 60.]
La media de los datos observados es en Bello: 31.43426294820717
El coeficiente de asimetría es en Bello:  -0.08276876109329274
El número de datos por intervalo es:
[ 94  83  85  86  78  89  91  89  93 107]
Los intervalos inferiores:
[ 0.  6. 12. 18. 24. 30. 36. 42. 48. 54.]
Los intervalos superiores:
[ 6. 12. 18. 24. 30. 36. 42. 48. 54. 60.]
La media de los datos observados es en Medellín: 30.781005586592183
El coeficiente de asimetría es en Medellí

In [144]:
# Bello
NDs=len(SkinThickness);NDb=len(SkinThickness1)
EPV=(sigmas*NDs+sigmab*NDb)/(NDs+NDb)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+ub*NDb)/(NDs+NDb)
VHM=((NDs*us**2+NDb*ub**2)/(NDs+NDb))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",ub)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 31.43426294820717
La credibilidad es: 0.6782793397499052


In [145]:
##Se evalúa la credibilidad de Sabaneta con Bello
#Se procede del valor esperado de la varianza
NDs=len(SkinThickness);NDm=len(SkinThickness1)
EPV=(sigmas*NDs+sigmam*NDm)/(NDs+NDm)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+um*NDm)/(NDs+NDm)
VHM=((NDs*us**2+NDm*um**2)/(NDs+NDm))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",um)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 30.781005586592183
La credibilidad es: 0.18383649183260908


In [146]:
##Se evalúa la credibilidad de Sabaneta con Medellín
#Se procede del valor esperado de la varianza
NDs=len(SkinThickness);NDe=len(SkinThickness2)
EPV=(sigmas*NDs+sigmae*NDe)/(NDs+NDe)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+ue*NDe)/(NDs+NDe)
VHM=((NDs*us**2+NDe*ue**2)/(NDs+NDe))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",ue)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 30.484848484848484
La credibilidad es: 0.6409395353147873


In [147]:
##Se evalúa la credibilidad de Sabaneta con Envigado
#Se procede del valor esperado de la varianza
NDs=len(SkinThickness);NDi=len(SkinThickness3)
EPV=(sigmas*NDs+sigmae*NDi)/(NDs+NDi)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+ui*NDi)/(NDs+NDi)
VHM=((NDs*us**2+NDi*ui**2)/(NDs+NDi))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",ui)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 30.01754385964912
La credibilidad es: 0.8814160179344548


In [148]:
##Se evalúa la credibilidad de Sabaneta con
#Se procede del valor esperado de la varianza
NDs=len(SkinThickness);NDi=len(SkinThickness4)
EPV=(sigmas*NDs+sigmae*NDi)/(NDs+NDi)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+ui*NDi)/(NDs+NDi)
VHM=((NDs*us**2+NDi*ui**2)/(NDs+NDi))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",ui)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 30.01754385964912
La credibilidad es: 0.8826688431151988


In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [ ]:
# Seleccionar las características médicas comunes para el análisis estadístico
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("Estadísticas Descriptivas (Medias y Desviaciones Estándar) por Sucursal Original y Datos Integrados\n")

# Estadísticas de Sabaneta (Original)
print("--- Sabaneta (Original) ---")
display(datos[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de Medellín (Original, antes de la integración de pacientes de Sabaneta)
print("\n--- Medellín (Original) ---")
display(datos2[medical_features].describe().loc[['mean', 'std']])

# Estadísticas de los Datos Integrados (Sabaneta + Medellín)
print("\n--- Datos Integrados (Sabaneta + Medellín) ---")
display(datos_integrados_Medellin[medical_features].describe().loc[['mean', 'std']])

In [149]:
##Se evalúa la credibilidad de Sabaneta con Envigado
#Se procede del valor esperado de la varianza
NDs=len(SkinThickness);NDi=len(SkinThickness5)
EPV=(sigmas*NDs+sigmae*NDi)/(NDs+NDi)

#Se procede con la estimación del Valor Hipotetico de la Media
uh=(us*NDs+ui*NDi)/(NDs+NDi)
VHM=((NDs*us**2+NDi*ui**2)/(NDs+NDi))-uh**2

#Se procede con la estimación del fc
fc=EPV/VHM

#Se procede con la estimación de la credibilidad
Cr=NDs/(NDs+fc)
print("La media de los datos internos es:",us)
print("La media de los datos externos es:",ui)
print("La credibilidad es:",Cr)

La media de los datos internos es: 30.94202898550725
La media de los datos externos es: 30.01754385964912
La credibilidad es: 0.8792064939476231


In [156]:
# Recalculando la credibilidad de Sabaneta con Medellín (usando SkinThickness2 para Medellín)
NDs_sabaneta = len(SkinThickness)
NDm_medellin = len(SkinThickness2)

#Se procede del valor esperado de la varianza
EPV_sm = (sigmas * NDs_sabaneta + sigmam * NDm_medellin) / (NDs_sabaneta + NDm_medellin)

#Se procede con la estimación del Valor Hipotetico de la Media
uh_sm = (us * NDs_sabaneta + um * NDm_medellin) / (NDs_sabaneta + NDm_medellin)
VHM_sm = ((NDs_sabaneta * us**2 + NDm_medellin * um**2) / (NDs_sabaneta + NDm_medellin)) - uh_sm**2

#Se procede con la estimación del fc
fc_sm = EPV_sm / VHM_sm

#Se procede con la estimación de la credibilidad
Cr_sm = NDs_sabaneta / (NDs_sabaneta + fc_sm)
print(f"La media de los datos internos (Sabaneta) es: {us}")
print(f"La media de los datos externos (Medellín) es: {um}")
print(f"La credibilidad de Sabaneta con Medellín es: {Cr_sm}")

La media de los datos internos (Sabaneta) es: 30.94202898550725
La media de los datos externos (Medellín) es: 30.781005586592183
La credibilidad de Sabaneta con Medellín es: 0.18063048162065917


In [157]:
# Recalculando la credibilidad de Sabaneta con Envigado (usando SkinThickness3 para Envigado)
NDs_sabaneta = len(SkinThickness)
NDe_envigado = len(SkinThickness3)

#Se procede del valor esperado de la varianza
EPV_se = (sigmas * NDs_sabaneta + sigmae * NDe_envigado) / (NDs_sabaneta + NDe_envigado)

#Se procede con la estimación del Valor Hipotetico de la Media
uh_se = (us * NDs_sabaneta + ue * NDe_envigado) / (NDs_sabaneta + NDe_envigado)
VHM_se = ((NDs_sabaneta * us**2 + NDe_envigado * ue**2) / (NDs_sabaneta + NDe_envigado)) - uh_se**2

#Se procede con la estimación del fc
fc_se = EPV_se / VHM_se

#Se procede con la estimación de la credibilidad
Cr_se = NDs_sabaneta / (NDs_sabaneta + fc_se)
print(f"La media de los datos internos (Sabaneta) es: {us}")
print(f"La media de los datos externos (Envigado) es: {ue}")
print(f"La credibilidad de Sabaneta con Envigado es: {Cr_se}")

La media de los datos internos (Sabaneta) es: 30.94202898550725
La media de los datos externos (Envigado) es: 30.484848484848484
La credibilidad de Sabaneta con Envigado es: 0.6451049594218039


In [158]:
# Recalculando la credibilidad de Sabaneta con Itaguí (usando SkinThickness4 para Itaguí)
NDs_sabaneta = len(SkinThickness)
NDi_itagui = len(SkinThickness4)

#Se procede del valor esperado de la varianza
EPV_si = (sigmas * NDs_sabaneta + sigmai * NDi_itagui) / (NDs_sabaneta + NDi_itagui)

#Se procede con la estimación del Valor Hipotetico de la Media
uh_si = (us * NDs_sabaneta + ui * NDi_itagui) / (NDs_sabaneta + NDi_itagui)
VHM_si = ((NDs_sabaneta * us**2 + NDi_itagui * ui**2) / (NDs_sabaneta + NDi_itagui)) - uh_si**2

#Se procede con la estimación del fc
fc_si = EPV_si / VHM_si

#Se procede con la estimación de la credibilidad
Cr_si = NDs_sabaneta / (NDs_sabaneta + fc_si)
print(f"La media de los datos internos (Sabaneta) es: {us}")
print(f"La media de los datos externos (Itaguí) es: {ui}")
print(f"La credibilidad de Sabaneta con Itaguí es: {Cr_si}")

La media de los datos internos (Sabaneta) es: 30.94202898550725
La media de los datos externos (Itaguí) es: 30.01754385964912
La credibilidad de Sabaneta con Itaguí es: 0.8822721565975485


In [159]:
# Recalculando la credibilidad de Sabaneta con Caldas (usando SkinThickness5 para Caldas)
NDs_sabaneta = len(SkinThickness)
NDc_caldas = len(SkinThickness5)

#Se procede del valor esperado de la varianza
EPV_sc = (sigmas * NDs_sabaneta + sigmac * NDc_caldas) / (NDs_sabaneta + NDc_caldas)

#Se procede con la estimación del Valor Hipotetico de la Media
uh_sc = (us * NDs_sabaneta + uc * NDc_caldas) / (NDs_sabaneta + NDc_caldas)
VHM_sc = ((NDs_sabaneta * us**2 + NDc_caldas * uc**2) / (NDs_sabaneta + NDc_caldas)) - uh_sc**2

#Se procede con la estimación del fc
fc_sc = EPV_sc / VHM_sc

#Se procede con la estimación de la credibilidad
Cr_sc = NDs_sabaneta / (NDs_sabaneta + fc_sc)
print(f"La media de los datos internos (Sabaneta) es: {us}")
print(f"La media de los datos externos (Caldas) es: {uc}")
print(f"La credibilidad de Sabaneta con Caldas es: {Cr_sc}")

La media de los datos internos (Sabaneta) es: 30.94202898550725
La media de los datos externos (Caldas) es: 30.518151815181522
La credibilidad de Sabaneta con Caldas es: 0.6028883872328537


In [160]:
credibility_scores = {
    'Bello': 0.6782793397499052, # From previous correct calculation
    'Medellín': Cr_sm,
    'Envigado': Cr_se,
    'Itaguí': Cr_si,
    'Caldas': Cr_sc
}

lowest_credibility_branch = min(credibility_scores, key=credibility_scores.get)
lowest_credibility_value = credibility_scores[lowest_credibility_branch]

print(f"La sucursal con la menor credibilidad para integrar con Sabaneta es: {lowest_credibility_branch} con una credibilidad de {lowest_credibility_value}")

La sucursal con la menor credibilidad para integrar con Sabaneta es: Medellín con una credibilidad de 0.18063048162065917


In [161]:
# Integrar los datos de Sabaneta y Medellín
datos_integrados_Medellin = pd.concat([datos, datos2], ignore_index=True)

print("Primeras 5 filas de los datos integrados (Sabaneta y Medellín):")
display(datos_integrados_Medellin.head())

print("Últimas 5 filas de los datos integrados (Sabaneta y Medellín):")
display(datos_integrados_Medellin.tail())

print(f"Número total de filas después de la integración: {len(datos_integrados_Medellin)}")

Primeras 5 filas de los datos integrados (Sabaneta y Medellín):


,ID,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch,Pregnancies
0,1.0,SAB-0001,196,101,0,321,31.010,0.352,80,0,Sabaneta,NaN
1,2.0,SAB-0002,93,115,17,366,27.789,1.846,31,0,Sabaneta,NaN
2,3.0,SAB-0003,95,73,37,135,44.594,1.461,63,1,Sabaneta,NaN
3,4.0,SAB-0004,94,91,19,191,23.219,1.303,22,0,Sabaneta,NaN
4,5.0,SAB-0005,189,47,22,600,24.747,0.993,43,1,Sabaneta,NaN


Últimas 5 filas de los datos integrados (Sabaneta y Medellín):


,ID,PatientID,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes,Branch,Pregnancies
1511,NaN,MED-0891,195,79,28,814,27.171,0.719,27,0,Medellín,15.0
1512,NaN,MED-0892,173,74,31,125,30.288,2.439,22,0,Medellín,13.0
1513,NaN,MED-0893,163,117,20,811,26.144,0.654,40,0,Medellín,15.0
1514,NaN,MED-0894,73,45,12,301,22.906,1.015,30,0,Medellín,8.0
1515,NaN,MED-0895,166,47,15,849,42.409,0.851,75,0,Medellín,10.0


Número total de filas después de la integración: 1516


In [162]:
# Seleccionar las características médicas comunes para el análisis
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Obtener las estadísticas descriptivas para la sede de Medellín (datos2) antes de la integración completa
medellin_profile_mean = datos2[medical_features].mean()
medellin_profile_std = datos2[medical_features].std()

print("Estadísticas de la sede de Medellín (medias):\n", medellin_profile_mean)
print("\nEstadísticas de la sede de Medellín (desviaciones estándar):\n", medellin_profile_std)

# Preparar los datos de Sabaneta para la asignación
sabaneta_patients_data = datos[medical_features]
print("\nPrimeras 5 filas de los datos de pacientes de Sabaneta para la asignación:\n")
display(sabaneta_patients_data.head())

Estadísticas de la sede de Medellín (medias):
 Glucose                     122.277095
BloodPressure                79.902793
SkinThickness                30.345251
Insulin                     421.836872
BMI                          32.879753
DiabetesPedigreeFunction      1.262841
Age                          51.626816
dtype: float64

Estadísticas de la sede de Medellín (desviaciones estándar):
 Glucose                      46.600008
BloodPressure                23.171246
SkinThickness                17.811288
Insulin                     251.093390
BMI                           9.976137
DiabetesPedigreeFunction      0.700308
Age                          17.737439
dtype: float64

Primeras 5 filas de los datos de pacientes de Sabaneta para la asignación:



,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,196,101,0,321,31.010,0.352,80
1,93,115,17,366,27.789,1.846,31
2,95,73,37,135,44.594,1.461,63
3,94,91,19,191,23.219,1.303,22
4,189,47,22,600,24.747,0.993,43


In [163]:
## Método 1: Valor de Pertenencia (Membership Value)

# Para cada paciente de Sabaneta, calculamos un 'valor de pertenencia' basado en cuántas de sus métricas
# están dentro de 1 desviación estándar de la media de Medellín.

def calculate_membership_value(patient_row, medellin_mean, medellin_std, features):
    membership_score = 0
    for feature in features:
        if (medellin_mean[feature] - medellin_std[feature] <= patient_row[feature] <= medellin_mean[feature] + medellin_std[feature]):
            membership_score += 1
    # Normalizamos la puntuación a un valor entre 0 y 1
    return membership_score / len(features)

sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Membership_Value']].head(10))

print("Distribución de los valores de pertenencia:\n")
print(sabaneta_patients_data['Membership_Value'].value_counts(normalize=True).sort_index())

Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):



/tmp/ipykernel_622/848920542.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Membership_Value
0,196,101,31.010,0.428571
1,93,115,27.789,0.714286
2,95,73,44.594,0.714286
3,94,91,23.219,0.857143
4,189,47,24.747,0.714286
5,185,58,33.292,0.714286
6,108,92,30.470,0.857143
7,191,110,34.060,0.428571
8,196,45,21.073,0.428571
9,163,62,29.303,0.857143


Distribución de los valores de pertenencia:

Membership_Value
0.000000    0.001610
0.142857    0.017713
0.285714    0.083736
0.428571    0.228663
0.571429    0.288245
0.714286    0.264090
0.857143    0.096618
1.000000    0.019324
Name: proportion, dtype: float64


In [164]:
## Método 2: Aceptación y Rechazo (Acceptance and Rejection)

# Clasificamos a los pacientes de Sabaneta como 'Aceptado' o 'Rechazado'
# si alguna de sus métricas está fuera de 1.5 desviaciones estándar de la media de Medellín.
# Un paciente 'Rechazado' podría necesitar atención especial o evaluación adicional.

def classify_patient(patient_row, medellin_mean, medellin_std, features):
    for feature in features:
        lower_bound = medellin_mean[feature] - 1.5 * medellin_std[feature]
        upper_bound = medellin_mean[feature] + 1.5 * medellin_std[feature]
        if not (lower_bound <= patient_row[feature] <= upper_bound):
            return 'Rechazado'
    return 'Aceptado'

sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Clasificación de pacientes de Sabaneta (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Acceptance_Status']].head(10))

print("Distribución del estado de aceptación:\n")
print(sabaneta_patients_data['Acceptance_Status'].value_counts())

Clasificación de pacientes de Sabaneta (primeras 10 filas):



/tmp/ipykernel_622/4146408618.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Acceptance_Status
0,196,101,31.010,Rechazado
1,93,115,27.789,Rechazado
2,95,73,44.594,Aceptado
3,94,91,23.219,Rechazado
4,189,47,24.747,Aceptado
5,185,58,33.292,Aceptado
6,108,92,30.470,Aceptado
7,191,110,34.060,Rechazado
8,196,45,21.073,Rechazado
9,163,62,29.303,Rechazado


Distribución del estado de aceptación:

Acceptance_Status
Rechazado    381
Aceptado     240
Name: count, dtype: int64


In [165]:
# Seleccionar las características médicas comunes para el análisis
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Obtener las estadísticas descriptivas para la sede de Medellín (datos2) antes de la integración completa
medellin_profile_mean = datos2[medical_features].mean()
medellin_profile_std = datos2[medical_features].std()

print("Estadísticas de la sede de Medellín (medias):\n", medellin_profile_mean)
print("\nEstadísticas de la sede de Medellín (desviaciones estándar):\n", medellin_profile_std)

# Preparar los datos de Sabaneta para la asignación
sabaneta_patients_data = datos[medical_features]
print("\nPrimeras 5 filas de los datos de pacientes de Sabaneta para la asignación:\n")
display(sabaneta_patients_data.head())

Estadísticas de la sede de Medellín (medias):
 Glucose                     122.277095
BloodPressure                79.902793
SkinThickness                30.345251
Insulin                     421.836872
BMI                          32.879753
DiabetesPedigreeFunction      1.262841
Age                          51.626816
dtype: float64

Estadísticas de la sede de Medellín (desviaciones estándar):
 Glucose                      46.600008
BloodPressure                23.171246
SkinThickness                17.811288
Insulin                     251.093390
BMI                           9.976137
DiabetesPedigreeFunction      0.700308
Age                          17.737439
dtype: float64

Primeras 5 filas de los datos de pacientes de Sabaneta para la asignación:



,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,196,101,0,321,31.010,0.352,80
1,93,115,17,366,27.789,1.846,31
2,95,73,37,135,44.594,1.461,63
3,94,91,19,191,23.219,1.303,22
4,189,47,22,600,24.747,0.993,43


In [166]:
## Método 1: Valor de Pertenencia (Membership Value)

# Para cada paciente de Sabaneta, calculamos un 'valor de pertenencia' basado en cuántas de sus métricas
# están dentro de 1 desviación estándar de la media de Medellín.

def calculate_membership_value(patient_row, medellin_mean, medellin_std, features):
    membership_score = 0
    for feature in features:
        if (medellin_mean[feature] - medellin_std[feature] <= patient_row[feature] <= medellin_mean[feature] + medellin_std[feature]):
            membership_score += 1
    # Normalizamos la puntuación a un valor entre 0 y 1
    return membership_score / len(features)

sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Membership_Value']].head(10))

print("Distribución de los valores de pertenencia:\n")
print(sabaneta_patients_data['Membership_Value'].value_counts(normalize=True).sort_index())

Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):



/tmp/ipykernel_622/848920542.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Membership_Value
0,196,101,31.010,0.428571
1,93,115,27.789,0.714286
2,95,73,44.594,0.714286
3,94,91,23.219,0.857143
4,189,47,24.747,0.714286
5,185,58,33.292,0.714286
6,108,92,30.470,0.857143
7,191,110,34.060,0.428571
8,196,45,21.073,0.428571
9,163,62,29.303,0.857143


Distribución de los valores de pertenencia:

Membership_Value
0.000000    0.001610
0.142857    0.017713
0.285714    0.083736
0.428571    0.228663
0.571429    0.288245
0.714286    0.264090
0.857143    0.096618
1.000000    0.019324
Name: proportion, dtype: float64


In [167]:
## Método 2: Aceptación y Rechazo (Acceptance and Rejection)

# Clasificamos a los pacientes de Sabaneta como 'Aceptado' o 'Rechazado'
# si alguna de sus métricas está fuera de 1.5 desviaciones estándar de la media de Medellín.
# Un paciente 'Rechazado' podría necesitar atención especial o evaluación adicional.

def classify_patient(patient_row, medellin_mean, medellin_std, features):
    for feature in features:
        lower_bound = medellin_mean[feature] - 1.5 * medellin_std[feature]
        upper_bound = medellin_mean[feature] + 1.5 * medellin_std[feature]
        if not (lower_bound <= patient_row[feature] <= upper_bound):
            return 'Rechazado'
    return 'Aceptado'

sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Clasificación de pacientes de Sabaneta (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Acceptance_Status']].head(10))

print("Distribución del estado de aceptación:\n")
print(sabaneta_patients_data['Acceptance_Status'].value_counts())

Clasificación de pacientes de Sabaneta (primeras 10 filas):



/tmp/ipykernel_622/4146408618.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Acceptance_Status
0,196,101,31.010,Rechazado
1,93,115,27.789,Rechazado
2,95,73,44.594,Aceptado
3,94,91,23.219,Rechazado
4,189,47,24.747,Aceptado
5,185,58,33.292,Aceptado
6,108,92,30.470,Aceptado
7,191,110,34.060,Rechazado
8,196,45,21.073,Rechazado
9,163,62,29.303,Rechazado


Distribución del estado de aceptación:

Acceptance_Status
Rechazado    381
Aceptado     240
Name: count, dtype: int64


In [168]:
# Seleccionar las características médicas comunes para el análisis
medical_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Obtener las estadísticas descriptivas para la sede de Medellín (datos2) antes de la integración completa
medellin_profile_mean = datos2[medical_features].mean()
medellin_profile_std = datos2[medical_features].std()

print("Estadísticas de la sede de Medellín (medias):\n", medellin_profile_mean)
print("\nEstadísticas de la sede de Medellín (desviaciones estándar):\n", medellin_profile_std)

# Preparar los datos de Sabaneta para la asignación
sabaneta_patients_data = datos[medical_features]
print("\nPrimeras 5 filas de los datos de pacientes de Sabaneta para la asignación:\n")
display(sabaneta_patients_data.head())

Estadísticas de la sede de Medellín (medias):
 Glucose                     122.277095
BloodPressure                79.902793
SkinThickness                30.345251
Insulin                     421.836872
BMI                          32.879753
DiabetesPedigreeFunction      1.262841
Age                          51.626816
dtype: float64

Estadísticas de la sede de Medellín (desviaciones estándar):
 Glucose                      46.600008
BloodPressure                23.171246
SkinThickness                17.811288
Insulin                     251.093390
BMI                           9.976137
DiabetesPedigreeFunction      0.700308
Age                          17.737439
dtype: float64

Primeras 5 filas de los datos de pacientes de Sabaneta para la asignación:



,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,196,101,0,321,31.010,0.352,80
1,93,115,17,366,27.789,1.846,31
2,95,73,37,135,44.594,1.461,63
3,94,91,19,191,23.219,1.303,22
4,189,47,22,600,24.747,0.993,43


In [169]:
## Método 1: Valor de Pertenencia (Membership Value)

# Para cada paciente de Sabaneta, calculamos un 'valor de pertenencia' basado en cuántas de sus métricas
# están dentro de 1 desviación estándar de la media de Medellín.

def calculate_membership_value(patient_row, medellin_mean, medellin_std, features):
    membership_score = 0
    for feature in features:
        if (medellin_mean[feature] - medellin_std[feature] <= patient_row[feature] <= medellin_mean[feature] + medellin_std[feature]):
            membership_score += 1
    # Normalizamos la puntuación a un valor entre 0 y 1
    return membership_score / len(features)

sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Membership_Value']].head(10))

print("Distribución de los valores de pertenencia:\n")
print(sabaneta_patients_data['Membership_Value'].value_counts(normalize=True).sort_index())

Pacientes de Sabaneta con su valor de pertenencia a Medellín (primeras 10 filas):



/tmp/ipykernel_622/848920542.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Membership_Value'] = sabaneta_patients_data.apply(lambda row: calculate_membership_value(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Membership_Value
0,196,101,31.010,0.428571
1,93,115,27.789,0.714286
2,95,73,44.594,0.714286
3,94,91,23.219,0.857143
4,189,47,24.747,0.714286
5,185,58,33.292,0.714286
6,108,92,30.470,0.857143
7,191,110,34.060,0.428571
8,196,45,21.073,0.428571
9,163,62,29.303,0.857143


Distribución de los valores de pertenencia:

Membership_Value
0.000000    0.001610
0.142857    0.017713
0.285714    0.083736
0.428571    0.228663
0.571429    0.288245
0.714286    0.264090
0.857143    0.096618
1.000000    0.019324
Name: proportion, dtype: float64


In [170]:
## Método 2: Aceptación y Rechazo (Acceptance and Rejection)

# Clasificamos a los pacientes de Sabaneta como 'Aceptado' o 'Rechazado'
# si alguna de sus métricas está fuera de 1.5 desviaciones estándar de la media de Medellín.
# Un paciente 'Rechazado' podría necesitar atención especial o evaluación adicional.

def classify_patient(patient_row, medellin_mean, medellin_std, features):
    for feature in features:
        lower_bound = medellin_mean[feature] - 1.5 * medellin_std[feature]
        upper_bound = medellin_mean[feature] + 1.5 * medellin_std[feature]
        if not (lower_bound <= patient_row[feature] <= upper_bound):
            return 'Rechazado'
    return 'Aceptado'

sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)

print("Clasificación de pacientes de Sabaneta (primeras 10 filas):\n")
display(sabaneta_patients_data[['Glucose', 'BloodPressure', 'BMI', 'Acceptance_Status']].head(10))

print("Distribución del estado de aceptación:\n")
print(sabaneta_patients_data['Acceptance_Status'].value_counts())

Clasificación de pacientes de Sabaneta (primeras 10 filas):



/tmp/ipykernel_622/4146408618.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sabaneta_patients_data['Acceptance_Status'] = sabaneta_patients_data.apply(lambda row: classify_patient(row, medellin_profile_mean, medellin_profile_std, medical_features), axis=1)


,Glucose,BloodPressure,BMI,Acceptance_Status
0,196,101,31.010,Rechazado
1,93,115,27.789,Rechazado
2,95,73,44.594,Aceptado
3,94,91,23.219,Rechazado
4,189,47,24.747,Aceptado
5,185,58,33.292,Aceptado
6,108,92,30.470,Aceptado
7,191,110,34.060,Rechazado
8,196,45,21.073,Rechazado
9,163,62,29.303,Rechazado


Distribución del estado de aceptación:

Acceptance_Status
Rechazado    381
Aceptado     240
Name: count, dtype: int64


**Resultados**

La sede de Medellín mostró la credibilidad más baja (aproximadamente 0.18) con Sabaneta, lo que indica que, de todas las opciones, es la que presenta el perfil de pacientes (basado en SkinThickness) más disímil a la sede que se cierra. Desde una perspectiva puramente estadística y de afinidad, esto la convierte en la opción con la menor similitud. Sin embargo, en un contexto de reubicación, una 'baja credibilidad' no necesariamente significa una 'mala opción', sino que implica una mayor divergencia en las características de los pacientes, lo que requerirá una gestión más cuidadosa.
Implicación Estratégica: La elección de Medellín como sede de integración implica que la EPS debe estar preparada para adaptar sus recursos y procesos en Medellín para atender a pacientes con perfiles que no se ajustan a su patrón habitual. Esto puede ser un desafío, pero también una oportunidad para expandir la capacidad y la experiencia de la sede.

Los pacientes con Membership_Value alto serán relativamente fáciles de integrar, ya que sus necesidades médicas se alinean bien con la capacidad y el conocimiento existente en Medellín. Para aquellos con valores bajos, la EPS necesitará considerar programas de transición específicos, consultas adicionales o incluso evaluar si otro centro con un perfil más adecuado podría atenderlos mejor.

La decisión de integrar la sede de Sabaneta en Medellín se basó en que Medellín presentó la menor credibilidad (0.18) con Sabaneta, según el análisis de SkinThickness. Esto, desde una perspectiva de afinidad estadística, indica que los perfiles de pacientes entre ambas sedes son los más disímiles. Esta divergencia es un punto crítico que el análisis de los métodos de reubicación confirma y profundiza.
cuántos pacientes (en este caso, 381, que representan más del 60%) caen en la categoría de 'Rechazados', la EPS puede cuantificar el desafío y planificar la asignación de recursos, personal especializado, o la necesidad de adaptar la infraestructura de Medellín de manera más efectiva.